# TP2 — Partie 2 : Construction et transformation des features RFM**TAISS 2026 — Filière F1 (Data Science)**Entrée : `data/processed/transactions_clean.parquet` (Partie 1).Sortie : une ligne par client, avec les features RFM brutes **et** transformées, prêtes pourle clustering de la Partie 3.Ce notebook répond directement aux **questions 2 et 3 du rapport** :- Q2 — pourquoi ne pas appliquer K-means sur les RFM bruts ?- Q3 — Fréquence et Montant sont corrélés : comment traiter cette redondance ?

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathfrom sklearn.preprocessing import StandardScalerfrom sklearn.decomposition import PCAfrom scipy import statspd.set_option("display.float_format", lambda v: f"{v:,.2f}")sns.set_theme(style="whitegrid")PROC = Path("../data/processed")FIG  = Path("../figures"); FIG.mkdir(parents=True, exist_ok=True)df = pd.read_parquet(PROC / "transactions_clean.parquet")print(f"{len(df):,} transactions | {df.CustomerID.nunique()} clients")df.head(3)

## 2.1 Date de référence (snapshot)La Récence se mesure par rapport à une date de référence. Le dataset s'arrête au 09/12/2011 :on prend **le lendemain de la dernière transaction** comme snapshot.**Pourquoi pas `today()` ?** Parce qu'on est en 2026 : tous les clients auraient ~15 ans derécence et la variable perdrait tout pouvoir discriminant. On se place au moment où l'analyseaurait été faite en conditions réelles. C'est un choix à annoncer explicitement — il conditionnetoute l'interprétation de R.

In [ ]:
SNAPSHOT = df.InvoiceDate.max() + pd.Timedelta(days=1)print("Dernière transaction :", df.InvoiceDate.max())print("Snapshot retenu      :", SNAPSHOT)

## 2.2 Agrégation par client| Feature | Définition | Choix à défendre ||---|---|---|| **Récence** | jours depuis le dernier achat | plus c'est bas, mieux c'est || **Fréquence** | nombre de **factures distinctes** | pas le nombre de lignes : une commande de 40 articles reste **un** acte d'achat || **Montant** | somme des `Amount` | CA total généré sur la période |On ajoute deux variables de contrôle, non utilisées pour le clustering mais utiles àl'interprétation en Partie 4 : `Anciennete` (jours depuis le **premier** achat) et`PanierMoyen` (Montant / Fréquence).

In [ ]:
rfm = df.groupby("CustomerID").agg(    Recence    = ("InvoiceDate", lambda s: (SNAPSHOT - s.max()).days),    Frequence  = ("Invoice", "nunique"),    Montant    = ("Amount", "sum"),    Anciennete = ("InvoiceDate", lambda s: (SNAPSHOT - s.min()).days),    NbArticles = ("Quantity", "sum"),).reset_index()rfm["PanierMoyen"] = rfm.Montant / rfm.Frequenceprint(rfm.shape)rfm.head()

In [ ]:
rfm[["Recence","Frequence","Montant","Anciennete","PanierMoyen"]].describe(    percentiles=[.25, .5, .75, .95, .99]).round(1)

### Premiers constats métier (à noter pour le rapport)

In [ ]:
mono = (rfm.Frequence == 1).mean()top1 = rfm.Montant.nlargest(int(len(rfm)*0.01)).sum() / rfm.Montant.sum()top10 = rfm.Montant.nlargest(int(len(rfm)*0.10)).sum() / rfm.Montant.sum()print(f"Clients mono-achat        : {mono:.1%}")print(f"Part du CA du top 1 %     : {top1:.1%}")print(f"Part du CA du top 10 %    : {top10:.1%}")print(f"Récence médiane           : {rfm.Recence.median():.0f} jours")print(f"Montant médian / moyen    : {rfm.Montant.median():,.0f} £ / {rfm.Montant.mean():,.0f} £")

> **Lecture** : ~28 % des clients n'ont acheté qu'une fois, et le top 10 % des clients pèse> ~64 % du CA. L'écart entre montant médian (~856 £) et moyen (~2 917 £) signale une> distribution très asymétrique : **la moyenne n'est pas un résumé fiable ici**. C'est> exactement le problème que K-means rencontrerait sur données brutes (section 2.4).

## 2.3 Distributions brutes — le diagnosticOn mesure l'asymétrie (skewness). Une distribution symétrique a un skew ≈ 0.

In [ ]:
RFM3 = ["Recence", "Frequence", "Montant"]print("Skewness (données brutes) :")print(rfm[RFM3].skew().round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))for i, c in enumerate(RFM3):    sns.histplot(rfm[c], bins=60, ax=axes[0, i], color="#4C72B0")    axes[0, i].set_title(f"{c} — brut (skew={rfm[c].skew():.1f})")    sns.boxplot(x=rfm[c], ax=axes[1, i], color="#4C72B0")plt.tight_layout(); plt.savefig(FIG / "02_distributions_brutes.png", dpi=150); plt.show()

## 2.4 ❗ Question 2 du rapport — pourquoi pas K-means sur les RFM bruts ?Quatre arguments, à donner dans cet ordre à l'oral :**1. K-means minimise une distance euclidienne → il est dominé par les grandes échelles.**Le Montant s'étale de 3 £ à 581 000 £, la Fréquence de 1 à 373, la Récence de 1 à 739.Sans mise à l'échelle, la distance entre deux clients est *presque entièrement* déterminéepar le Montant : la Récence et la Fréquence ne pèsent quasiment rien. La segmentation« RFM » serait en réalité une segmentation « M ».**2. Skewness extrême (Montant ≈ 25, Fréquence ≈ 12).**Une poignée de grossistes est si éloignée du nuage que K-means leur consacre des clustersd'une poignée d'individus, pendant que les 95 % de clients ordinaires sont écrasés dans unseul gros cluster indifférencié — donc inutilisable pour le marketing.**3. K-means suppose des clusters sphériques et de variance comparable.**Une distribution log-normale produit des groupes allongés : l'hypothèse est violée.**4. L'écart pertinent est multiplicatif, pas additif.**Passer de 100 £ à 200 £ n'a pas le même sens métier que passer de 10 000 £ à 10 100 £ —même écart absolu, comportements sans rapport. Le log convertit les rapports en écarts :`log(200) − log(100) = log(10 100/10 000)`… non, précisément l'inverse — le log rend l'écart100→200 (×2) **plus grand** que 10 000→10 100 (×1,01), ce qui est bien le sens métier voulu.**Ce que la transformation apporte** : on le mesure, on ne l'affirme pas.

In [ ]:
rfm_log = rfm[RFM3].apply(np.log1p)rfm_log.columns = [f"log_{c}" for c in RFM3]comp = pd.DataFrame({    "skew_brut": rfm[RFM3].skew().values,    "skew_log":  rfm_log.skew().values,    "ratio_max/median_brut": (rfm[RFM3].max() / rfm[RFM3].median()).values,}, index=RFM3).round(2)display(comp)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))for i, c in enumerate(RFM3):    sns.histplot(np.log1p(rfm[c]), bins=60, ax=axes[i], color="#55A868")    axes[i].set_title(f"log1p({c}) — skew={np.log1p(rfm[c]).skew():.2f}")plt.tight_layout(); plt.savefig(FIG / "02_distributions_log.png", dpi=150); plt.show()

> **À écrire dans le rapport** : la transformation `log1p` fait passer la skewness du Montant> de **25,3 à 0,27** et celle de la Fréquence de **12,0 à 1,00** — les distributions deviennent> quasi-normales. On utilise `log1p` (= log(1+x)) plutôt que `log` par sécurité numérique, même> si après nettoyage aucune valeur n'est nulle.>> **Pourquoi le log ne suffit pas et qu'il faut ensuite standardiser** : après log, les trois> variables restent sur des plages différentes (log-Récence ~0–6,6 ; log-Montant ~1–13). La> standardisation (moyenne 0, écart-type 1) leur donne **un poids égal** dans la distance> euclidienne — ce qui est le choix explicite : on considère que R, F et M comptent autant l'un> que l'autre. Un autre choix (pondérer M davantage) serait défendable, mais il doit être assumé.

## 2.5 Standardisation

In [ ]:
scaler = StandardScaler()X = scaler.fit_transform(rfm_log)X = pd.DataFrame(X, columns=["R_scaled", "F_scaled", "M_scaled"], index=rfm.index)print("Moyennes :", X.mean().round(6).values, " | Écarts-types :", X.std().round(4).values)X.describe().round(2)

### Vérification visuelle : avant / après

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))sns.scatterplot(x=rfm.Frequence, y=rfm.Montant, s=8, alpha=.4, ax=axes[0])axes[0].set_title("Brut — quelques points écrasent tout le reste")sns.scatterplot(x=X.F_scaled, y=X.M_scaled, s=8, alpha=.4, ax=axes[1], color="#55A868")axes[1].set_title("log + standardisé — structure lisible")plt.tight_layout(); plt.savefig(FIG / "02_avant_apres_transformation.png", dpi=150); plt.show()

## 2.6 ❗ Question 3 du rapport — la redondance Fréquence / MontantOn mesure la corrélation **avant et après** transformation. Attention au piège :la corrélation de Pearson sur données brutes est trompeuse (elle est écrasée par les outliers).

In [ ]:
print("Pearson — brut :");        display(rfm[RFM3].corr().round(2))print("Pearson — log :");         display(rfm_log.corr().round(2))print("Spearman (rangs, robuste aux outliers) :"); display(rfm[RFM3].corr(method="spearman").round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))sns.heatmap(rfm[RFM3].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax[0])ax[0].set_title("Corrélations — brut")sns.heatmap(rfm_log.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax[1])ax[1].set_title("Corrélations — log")plt.tight_layout(); plt.savefig(FIG / "02_correlations.png", dpi=150); plt.show()

**Constat** : après log, corr(Fréquence, Montant) = **0,85** (Spearman 0,86). C'est très élevé— logique : qui commande souvent dépense plus. La corrélation brute n'était que de 0,62,artificiellement basse parce que les outliers cassaient la relation linéaire.**Conséquence sur K-means** : deux variables corrélées à 0,85 pointent presque dans la mêmedirection. La distance euclidienne compte donc « l'intensité d'achat » **deux fois** et laRécence une seule → l'axe R est implicitement sous-pondéré.### Quatre options, et celle qu'on retient| Option | Effet | Verdict ||---|---|---|| **A. Ne rien faire (garder R, F, M)** | F et M pèsent double | ✔️ **retenu** pour le socle || B. Supprimer M, garder R + F | perte de l'information « valeur » | ✘ M est la variable la plus parlante pour le marketing || C. Remplacer M par le **panier moyen** (M/F) | décorrèle mais change le sens : mesure la valeur *par commande*, pas la valeur totale | ⚠️ testé en annexe || D. **PCA** sur les 3 variables log-standardisées | décorrèle par construction | ⚠️ testé ci-dessous |**Justification du choix A** : l'objectif est une segmentation **interprétable par le marketing**.Les centroïdes doivent se lire directement en « ce client achète souvent / dépense beaucoup /est parti depuis longtemps ». Après une PCA, les centroïdes s'expriment en composantesabstraites, et il faut un aller-retour pour les retraduire — on perd en défendabilité à l'oralce qu'on gagne en orthogonalité. La redondance F/M est donc **assumée et documentée**, pasignorée : on sait que la segmentation obtenue est principalement structurée parl'axe « valeur client », ce qui est précisément ce que veut la direction.On vérifie quand même ce que dirait la PCA :

In [ ]:
pca = PCA().fit(X)print("Variance expliquée :", pca.explained_variance_ratio_.round(3))print("Cumulée            :", pca.explained_variance_ratio_.cumsum().round(3))display(pd.DataFrame(pca.components_, columns=RFM3,                     index=[f"PC{i+1}" for i in range(3)]).round(2))

> **Lecture** : PC1 capte **76 %** de la variance et charge positivement F et M, négativement R> → c'est un axe **« valeur / engagement client »**. PC2 (19 %) est porté par la Récence.> Les deux premières composantes couvrent 95 % de l'information : la structure est> essentiellement **bidimensionnelle**, ce qui confirme la redondance F/M.>> → À citer en Partie 3 pour justifier qu'un nombre modéré de clusters (4 à 6) suffit :> il n'y a pas 8 directions indépendantes à découper.

### Variante C (annexe) — panier moyen au lieu du montant total

In [ ]:
alt = pd.DataFrame({    "log_R": np.log1p(rfm.Recence),    "log_F": np.log1p(rfm.Frequence),    "log_PanierMoyen": np.log1p(rfm.PanierMoyen),})print("Corrélation F / PanierMoyen (log) :",      round(alt.log_F.corr(alt.log_PanierMoyen), 2))display(alt.corr().round(2))

> Le panier moyen est bien moins corrélé à la Fréquence que le Montant total.> Cette variante est conservée comme **test de robustesse** (Partie 3 / extension 3) :> si les segments obtenus sont similaires, la conclusion est solide.

## 2.7 Scores RFM par quintiles (grille classique, en complément)En parallèle du clustering, on calcule la grille RFM « métier » classique : chaque dimensionnotée de 1 à 5 par quintiles. Elle sert de **garde-fou d'interprétation** en Partie 4 —si un cluster K-means ne correspond à aucune combinaison de scores lisible, c'est un signald'alerte.Attention : la Récence s'inverse (récence faible = bon score = 5).

In [ ]:
rfm["R_score"] = pd.qcut(rfm.Recence, 5, labels=[5, 4, 3, 2, 1]).astype(int)rfm["F_score"] = pd.qcut(rfm.Frequence.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)rfm["M_score"] = pd.qcut(rfm.Montant, 5, labels=[1, 2, 3, 4, 5]).astype(int)rfm["RFM_score"] = rfm.R_score.astype(str) + rfm.F_score.astype(str) + rfm.M_score.astype(str)rfm["RFM_somme"] = rfm[["R_score","F_score","M_score"]].sum(axis=1)print("Note : la Fréquence a beaucoup d'ex-aequo (28 % de clients à F=1),")print("d'où le rank(method='first') pour forcer des quintiles de tailles égales.")display(rfm.RFM_somme.value_counts().sort_index())rfm[["CustomerID","Recence","Frequence","Montant","R_score","F_score","M_score","RFM_score"]].head()

## 2.8 SauvegardeOn sauvegarde trois objets :- `rfm_features.parquet` : les features brutes + scores (pour l'interprétation, Partie 4) ;- `rfm_scaled.parquet`   : la matrice log-standardisée (entrée du clustering, Partie 3) ;- `scaler.joblib`        : le scaler ajusté, pour pouvoir **retransformer les centroïdes**  en unités métier en Partie 4 (`scaler.inverse_transform` puis `np.expm1`).Ce dernier point est important : sans le scaler, les centroïdes de K-means restentd'illisibles valeurs standardisées.

In [ ]:
import joblibX_full = X.copy()X_full.insert(0, "CustomerID", rfm.CustomerID.values)rfm.to_parquet(PROC / "rfm_features.parquet", index=False)X_full.to_parquet(PROC / "rfm_scaled.parquet", index=False)joblib.dump({"scaler": scaler, "snapshot": SNAPSHOT, "cols": RFM3}, PROC / "scaler.joblib")print("✅ Sauvegardé :", [p.name for p in PROC.glob('rfm*')] + ["scaler.joblib"])print("Matrice de clustering :", X_full.shape)

### (Optionnel) Refaire la même chose sur le jeu net des retoursPour préparer la comparaison de la question 1, on recalcule les RFM sur`transactions_clean_net.parquet`. Le clustering de la Partie 3 tournera sur les deux et oncomparera les partitions par ARI.

In [ ]:
df_net = pd.read_parquet(PROC / "transactions_clean_net.parquet")rfm_net = df_net.groupby("CustomerID").agg(    Recence   = ("InvoiceDate", lambda s: (SNAPSHOT - s.max()).days),    Frequence = ("Invoice", "nunique"),    Montant   = ("Amount", "sum"),).reset_index()# un retour peut rendre le CA net négatif ou nul → log impossible : on trace ces casneg = (rfm_net.Montant <= 0).sum()print(f"Clients à CA net <= 0 après retours : {neg} ({neg/len(rfm_net):.2%}) — exclus du clustering net")rfm_net = rfm_net[rfm_net.Montant > 0]Xn = StandardScaler().fit_transform(rfm_net[RFM3].apply(np.log1p))pd.DataFrame(Xn, columns=["R_scaled","F_scaled","M_scaled"]).assign(    CustomerID=rfm_net.CustomerID.values).to_parquet(PROC / "rfm_scaled_net.parquet", index=False)print("✅ rfm_scaled_net.parquet :", Xn.shape)

## Mini-résumé (réutilisable dans le rapport)> Les 776 582 transactions nettoyées sont agrégées en **5 852 clients**, décrits par la Récence> (jours depuis le dernier achat), la Fréquence (nombre de **factures distinctes**, et non de> lignes) et le Montant (CA cumulé), au snapshot du 10/12/2011 — lendemain de la dernière> transaction, retenu plutôt que la date du jour pour préserver le pouvoir discriminant de R.> Les distributions brutes sont fortement asymétriques (skew de 25,3 pour le Montant, 12,0 pour> la Fréquence) et d'échelles incomparables : appliquer K-means directement reviendrait à> segmenter sur le seul Montant et à isoler une poignée de grossistes. Une transformation> `log1p` ramène la skewness à 0,27 et 1,00, puis une standardisation donne un poids égal aux> trois dimensions. Après transformation, Fréquence et Montant restent corrélées à **0,85** :> cette redondance est assumée — une PCA (PC1 = 76 % de la variance, axe « valeur client »)> confirme une structure essentiellement bidimensionnelle, mais on conserve R, F, M bruts pour> garder des centroïdes directement interprétables par le marketing. Deux variantes (panier> moyen, RFM net des retours) sont préparées pour tester la robustesse en Partie 3.**Questions du rapport traitées ici** : Q2 (log + standardisation) ✅ · Q3 (redondance F/M) ✅**Pour la Partie 5** : le choix du snapshot et la pondération égale de R, F, M sont desdécisions d'analyste, pas des vérités — à mentionner dans les limites.